# 03 — Portable records and leakage-safe splits

**Estimated time:** 40 minutes<br>
**Prerequisites:** 02 — Dataset exploration and validation<br>
**Learner-produced evidence:** split counts, stable IDs, hashes, and a zero-leakage report

## Learning objectives

- Trace immutable source rows into framework-neutral chat records.
- Explain group-aware balancing and stable record identifiers.
- Verify split integrity without inspecting frozen-test examples.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

Evaluation is credible only when development examples and final-exam examples are genuinely independent. Support datasets often contain templates, paraphrases, repeated conversations, or records from the same source. A random row split can place siblings on both sides and reward memorization. Stable IDs, grouping, manifests, and leakage gates create evidence boundaries.

## Key terms in plain language

- **example ID:** a stable identifier derived from learning-relevant content or a governed source key.
- **split:** an explicit assignment of records to train, validation, or test.
- **group:** records that must travel together because they share a source, template, subject, or near-duplicate signal.
- **stratification:** preserving important class proportions across splits when the grouping constraints permit it.
- **data leakage:** information crossing an evidence boundary in a way that makes measured performance unrealistically easy.
- **frozen test set:** a held-out split whose examples and results are not used to design the same evaluated change.
- **manifest:** a machine-readable record of split membership, counts, configuration, and fingerprints.
- **fingerprint:** a deterministic digest used to verify that an artifact or evaluation input has not silently changed.


## Mental model — how to think about this

Picture three locked rooms. The training room may teach. The validation room may help choose between already-defined options. The test room opens only after the choice is locked. Related examples are a family and must enter the same room. A manifest is the signed seating chart; a fingerprint is the tamper seal.

### Running example

All password-reset paraphrases assigned to one duplicate/template group travel together into train, validation, or test. If one appears in train and a sibling appears in test, the score may reward memorized wording. If you inspect a test mistake and rewrite the prompt, that test has become development data.

### Questions to ask before continuing

- What makes two records related enough that seeing one helps predict the other?
- Which grouping key should take priority over perfect label balance?
- Has any test content, label, error, or result influenced prompts, examples, thresholds, or training?
- Can the exact split be reconstructed and checked without trusting filenames?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Group before assigning splits.** Exact duplicates, near duplicates, shared templates, users, documents, or time windows should remain together when they can transmit learning signal.
- **Use stable IDs and versioned configuration.** Record normalization rules, group logic, seed, target proportions, and the code/data version that produced the manifest.
- **Automate leakage gates.** Assert disjoint IDs and group keys, verify file fingerprints, and fail the pipeline when a protected boundary is violated.
- **Use validation for iteration and test for the final estimate.** After looking at test errors, the next fix needs a newly defined change and a fresh untouched evaluation version.
- **Prefer honest imbalance to broken independence.** Group constraints can prevent exact class ratios; document the resulting slices instead of splitting a related group.

## Common mistakes and why they fail

- **Randomly splitting rows with a fixed seed.** Reproducibility does not prevent related content from leaking across splits.
- **Using test examples as few-shot demonstrations.** Their labels have now entered the system being evaluated.
- **Tuning after reading test errors.** The original test result becomes development feedback, not an untouched estimate.
- **Claiming `no leakage` from one detector.** A passed check means no violation was detected under the implemented definition; undocumented semantic relationships may remain.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [scikit-learn common pitfalls: data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)
- **Tool guidance:** [scikit-learn grouped splitting documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Evidence boundaries

| Split | May influence | Must not influence |
|---|---|---|
| Train | Learned weights, train-derived statistics, demonstrations | Final generalization claim |
| Validation | Prompt, checkpoint, threshold, and configuration choice | Learned examples or final claim |
| Frozen test | Final measurement of already-locked methods | Any earlier design choice |

“Frozen” is a governance promise, not a file permission. A hash reveals
changed bytes but does not stop a person from reading them. This notebook
reads manifest metadata and runs automated checks, but does not display or
use test examples. If a test result later causes a prompt change, that test
has become development data and a fresh untouched boundary is required.


In [ ]:
import json

from aai_local_finetuning.data import (
    check_split_files,
    text_similarity,
    verify_manifest,
)
from aai_local_finetuning.evaluation import load_records_jsonl
from aai_local_finetuning.settings import load_settings

settings = load_settings()
processed = settings.processed_dir
manifest = json.loads((processed / "manifest.json").read_text(encoding="utf-8"))
train = tuple(load_records_jsonl(processed / "train.jsonl"))
validation = tuple(load_records_jsonl(processed / "valid.jsonl"))
split_contract = {
    name: {
        "records": descriptor["record_count"],
        "frozen": descriptor["frozen"],
        "sha256": descriptor["sha256"],
    }
    for name, descriptor in manifest["splits"].items()
}
verification = verify_manifest(processed / "manifest.json")
{
    "splits": split_contract,
    "artifact_hashes_valid": verification.valid,
    "checked_files": verification.checked_files,
    "mismatches": verification.mismatches,
}

## What one portable record contains

Framework-neutral JSONL preserves identity, source version, messages,
labels, grouping evidence, flags, and difficulty. Source responses are
not copied as training targets; a versioned response policy renders a
short target independently. The preview is already masked and bounded.


In [ ]:
example = train[0]
safe_preview = {
    "example_id": example.example_id,
    "input_preview": example.input_text[:160],
    "target": {
        "intent": example.target.intent,
        "category": example.target.category,
        "requires_escalation": example.target.requires_escalation,
        "response_preview": example.target.response[:100],
    },
    "flags": example.flags,
    "difficulty": example.difficulty,
    "split_group": example.metadata.get("split_group"),
}
safe_preview

## Balance and separation

The source has no reliable conversation, account, document, template,
or timestamp identifier. The pipeline therefore keeps inferred exact,
template, and near-duplicate groups together, excludes label-conflicting
groups, and records this limitation instead of inventing source fields.


In [ ]:
train_intents = {}
validation_intents = {}
for record in train:
    train_intents[record.target.intent] = train_intents.get(record.target.intent, 0) + 1
for record in validation:
    validation_intents[record.target.intent] = (
        validation_intents.get(record.target.intent, 0) + 1
    )
balance = {
    "train_unique_counts": sorted(set(train_intents.values())),
    "validation_unique_counts": sorted(set(validation_intents.values())),
    "train_validation_id_overlap": len(
        {item.example_id for item in train} & {item.example_id for item in validation}
    ),
    "dataset_fingerprint": manifest["dataset_fingerprint"],
}
balance

## Run the automated leakage gate

The gate distinguishes exact overlap, inferred-template overlap, near
duplicates, shared source groups, target leakage, and demonstration
leakage. Development contamination—changing a method after test feedback—is
a process failure and cannot be detected from files alone.

A passing gate means no configured relationship crossed the prepared
boundaries. It does not prove that heuristic grouping discovered every
semantic relationship in the world.


In [ ]:
leakage = check_split_files(processed)
{
    "passed": leakage.passed,
    "finding_count": len(leakage.findings),
    "findings_by_kind": leakage.counts,
}

## Exercise — reason about near duplicates

Change the second invented sentence and observe the similarity. Decide
whether a numeric threshold is sufficient evidence of common origin.
Success means you record both a decision and a limitation.


In [ ]:
sentence_a = "I forgot my password and cannot sign in"
sentence_b = "I cannot sign in because I forgot the password"
similarity = text_similarity(sentence_a, sentence_b)
threshold = manifest["processing"]["near_duplicate_threshold"]
grouping_decision = similarity >= threshold
limitation = (
    "Similarity is a reproducible heuristic; it is not a verified "
    "conversation or template identifier from the source."
)
{
    "similarity": round(similarity, 3),
    "configured_threshold": threshold,
    "same_group_at_threshold": grouping_decision,
    "limitation": limitation,
}

**Hint:** a threshold makes behavior repeatable, not automatically true.
Group conservatively and preserve the documented uncertainty.


## Checkpoint

You can identify what each split is allowed to influence and explain
why a frozen flag, file hash, stable IDs, and leakage test work together.

**Next:** `04_deterministic_baselines.ipynb` establishes a sanity floor
and a transparent meaningful baseline using train and validation only.
